# Healthcare Knowledge Graph V6 — Evidence-grounded Question Answering

**Research prototype.** Loads V5.1 output ZIP, keeps the 293-node clinical KG read-only, performs deterministic intent routing, returns provenance-linked NHANES screening estimates, retrieves baseline KG nodes, and blocks unsupported treatment-eligibility and cross-cycle trend claims. This is **not** an LLM agent or clinical decision support system. Clinical mappings remain unapproved.

In [ ]:
from pathlib import Path
INPUT_ZIP = Path('/mnt/data/Healthcare_KG_V5_1_Outputs.zip')
if not INPUT_ZIP.exists():
    INPUT_ZIP = Path('Healthcare_KG_V5_1_Outputs.zip')
if not INPUT_ZIP.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        INPUT_ZIP = Path(next(k for k in uploaded if k.endswith('.zip')))
    except ImportError:
        raise FileNotFoundError('Place Healthcare_KG_V5_1_Outputs.zip in the working directory.')

In [ ]:

import csv, hashlib, io, json, re, zipfile, datetime, pathlib
from collections import Counter

OUTPUT_DIR = pathlib.Path('Healthcare_KG_V6_Outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
INPUT_ZIP = pathlib.Path(INPUT_ZIP)
assert INPUT_ZIP.is_file(), f'Missing input archive: {INPUT_ZIP}'

with zipfile.ZipFile(INPUT_ZIP) as z:
    needed = ['baseline_kg_COPY.json','population_kg_layer_v5_1.json','population_observations_v5_1.json','clinical_link_adjudication_v5_1.json','v5_1_validation_report.json']
    for filename in needed:
        assert filename in z.namelist(), f'Missing {filename}'
    raw_baseline = z.read('baseline_kg_COPY.json')
    baseline = json.loads(raw_baseline)
    layer = json.loads(z.read('population_kg_layer_v5_1.json'))
    observations = json.loads(z.read('population_observations_v5_1.json'))
    previous_report = json.loads(z.read('v5_1_validation_report.json'))
    csv_links = list(csv.DictReader(io.StringIO(z.read('clinical_link_adjudication_v5_1.csv').decode('utf-8-sig'))))

baseline_sha = hashlib.sha256(raw_baseline).hexdigest()
assert baseline_sha == layer['baseline_sha256'] == previous_report['baseline_sha256'], 'Baseline checksum mismatch'
assert len(baseline['nodes']) == 293 and len(baseline['edges']) == 517, 'Unexpected baseline graph size'
assert len(observations) == 4, 'Expected 4 population observations'
assert all(o.get('epidemiological_validation') == 'REVIEW_REQUIRED' for o in observations)
assert all(not o.get('trend_comparison_allowed', True) for o in observations)
assert all(not o.get('treatment_eligibility_estimate', True) for o in observations)

# Do not promote any lexical or rule-based concept link to clinical evidence.
concept_links = layer['curated_conceptual_links']
assert all(x.get('human_review_completed') is False for x in concept_links)
assert all(x.get('relation') == 'SCREENING_PROXY_RELATED_TO' for x in concept_links)
node_lookup = {str(n[0]): n for n in baseline['nodes']}

# Scope-restricted deterministic retrieval. No model-generated clinical facts.
STOPWORDS = set('a an the and or of to is are what which how many in for on with does do about us united states patients adults explain tell me evidence research'.split())
def tokens(s):
    return set(re.findall(r'[a-z0-9]+', str(s).lower())) - STOPWORDS

def clinical_search(question, limit=6):
    q = tokens(question)
    if not q:
        return []
    hits = []
    for node in baseline['nodes']:
        desc = str(node[2])
        name = str(node[0])
        overlap = q & tokens(desc + ' ' + name)
        if overlap:
            hits.append({'node_id': str(node[0]), 'type': node[1], 'description': desc[:320], 'match_terms': sorted(overlap), 'score': len(overlap)})
    return sorted(hits, key=lambda x: (-x['score'], x['node_id']))[:limit]

def latest_primary():
    primaries = [o for o in observations if o['scenario'] == 'primary']
    return max(primaries, key=lambda o: o['period'])

def classify(question):
    q = question.lower()
    if re.search(r'\b(mars|moon|jupiter|venus)\b', q):
        return 'out_of_scope'
    if re.search(r'eligible|eligibility|prescri(b|ption)|contraindicat|addressable market|revenue|market size|tam|how many.*(kerendia|finerenone)',q):
        return 'eligibility_guard'
    if re.search(r'\btrend\b|increas|decreas|change over time|growth|declin|compar.*(years|cycles|periods)',q):
        return 'trend_guard'
    if re.search(r'\b(ckd|kidney|renal|diabet|population|nhanes|prevalence)\b',q) and re.search(r'how many|number|population|prevalence|estimate|count',q):
        return 'population'
    if re.search(r'\b(ckd|kidney|renal|diabet|finerenone|kerendia|heart failure|albuminuria)\b',q):
        return 'clinical_retrieval'
    return 'out_of_scope'

def respond(question):
    route = classify(question)
    trace = [{'step':'intent_router','route':route}]
    evidence = []
    if route == 'eligibility_guard':
        answer = ('BLOCKED: These NHANES observations are single-visit CKD laboratory screening plus probable T2D proxies; they do not establish Kerendia eligibility, contraindications, treatment utilization, or addressable market. No eligible-patient count can be derived from this layer.')
        status = 'BLOCKED_UNSUPPORTED_ELIGIBILITY'
    elif route == 'trend_guard':
        answer = ('BLOCKED: A population trend cannot be inferred until serum-creatinine and laboratory comparability, differing NHANES survey periods, and missing-data assumptions are validated. Cross-cycle numbers are not evidence of a trend.')
        status = 'BLOCKED_UNVALIDATED_TREND'
    elif route == 'population':
        o = latest_primary()
        lo,hi = o['count_ci_95']
        answer = (f"NHANES {o['period']} ({o['scenario']}) estimates {o['estimated_adults']:,.0f} U.S. adults (reported 95% CI {lo:,.0f}–{hi:,.0f}) meeting the notebook-defined single-visit CKD screen and probable T2D proxy. Complete-case screening prevalence: {100*o['screening_prevalence_complete_case']:.2f}%. Weight: {o['survey_weight']}. Epidemiological review is required. This is not confirmed disease, treatment eligibility, or a validated trend.")
        evidence = [{'kind':'population_observation','id':o['id'],'source':o['provenance']['source'],'period':o['period'],'scenario':o['scenario'],'validation':o['epidemiological_validation']}]
        trace.append({'step':'population_lookup','observation_id':o['id']})
        status = 'SCREENING_ESTIMATE_WITH_CAVEATS'
    elif route == 'clinical_retrieval':
        hits = clinical_search(question)
        trace.append({'step':'clinical_kg_lexical_retrieval','hits':len(hits)})
        if hits:
            answer = 'Relevant nodes found in the baseline clinical KG (retrieval only, not an independently validated clinical answer): ' + '; '.join(f"{h['node_id']}: {h['description']}" for h in hits[:4]) + '. These matches do not establish clinical efficacy or causal evidence; inspect underlying publications before drawing conclusions.'
            evidence = [{'kind':'baseline_kg_node','id':h['node_id'],'baseline_sha256':baseline_sha,'description':h['description']} for h in hits[:4]]
            status = 'RETRIEVAL_ONLY_NOT_CLINICALLY_VERIFIED'
        else:
            answer = 'No matching baseline KG nodes were retrieved. No clinical claim is generated.'
            status = 'NO_EVIDENCE_RETRIEVED'
    else:
        answer = 'Out of scope for the available clinical KG and NHANES population observations. No answer is fabricated.'
        status = 'OUT_OF_SCOPE'
    return {'question':question,'route':route,'status':status,'answer':answer,'evidence':evidence,'trace':trace,'limitations':['Epidemiological validation pending','Human review of 16 concept mappings pending','Cross-cycle serum creatinine comparability pending']}

QUESTIONS = [
    'How many US adults have CKD and diabetes?',
    'What is the estimated prevalence of CKD and probable T2D?',
    'How many patients are eligible for Kerendia?',
    'What is the addressable market for finerenone?',
    'What is the CKD trend across cycles?',
    'Explain chronic kidney disease evidence in the KG',
    'What does the KG contain about finerenone?',
    'What is the population of Mars?'
]
results = [respond(q) for q in QUESTIONS]

# Regression suite exercises evidence boundaries as well as expected routes.
tests = {}
def check(name, condition):
    tests[name] = bool(condition)
    if not condition:
        raise AssertionError(name)
check('baseline_checksum', hashlib.sha256(raw_baseline).hexdigest() == previous_report['baseline_sha256'])
check('baseline_node_count', len(baseline['nodes']) == 293)
check('baseline_edge_count', len(baseline['edges']) == 517)
check('four_population_observations', len(observations) == 4)
check('no_clinical_link_human_approval', all(not x.get('human_review_completed') for x in concept_links))
check('population_has_observation_id', results[0]['evidence'][0]['id'] == latest_primary()['id'])
check('population_has_confidence_interval', '95% CI' in results[0]['answer'])
check('population_reports_review_required', 'review is required' in results[0]['answer'])
check('population_weight_is_explicit', latest_primary()['survey_weight'] in results[0]['answer'])
check('eligibility_blocked', results[2]['status'] == 'BLOCKED_UNSUPPORTED_ELIGIBILITY')
check('market_sizing_blocked', results[3]['status'] == 'BLOCKED_UNSUPPORTED_ELIGIBILITY')
check('trend_blocked', results[4]['status'] == 'BLOCKED_UNVALIDATED_TREND')
check('clinical_not_misrepresented_as_validated', results[5]['status'] in ('RETRIEVAL_ONLY_NOT_CLINICALLY_VERIFIED','NO_EVIDENCE_RETRIEVED'))
check('out_of_scope_guard', results[7]['status'] == 'OUT_OF_SCOPE')
check('bridge_not_primary', latest_primary()['scenario'] == 'primary')
check('no_unreviewed_concept_promotion', all(x.get('mapping_status') == 'CANDIDATE_REVIEW_REQUIRED' for x in concept_links))
check('baseline_source_unmodified', hashlib.sha256(raw_baseline).hexdigest() == baseline_sha)

# Save standalone evidence artifacts and compact audit trail.
(OUTPUT_DIR/'v6_qa_examples.json').write_text(json.dumps(results,indent=2,ensure_ascii=False))
(OUTPUT_DIR/'v6_population_observations.json').write_text(json.dumps(observations,indent=2))
(OUTPUT_DIR/'v6_unapproved_concept_links.json').write_text(json.dumps(concept_links,indent=2))
(OUTPUT_DIR/'v6_clinical_kg_baseline_sha256.txt').write_text(baseline_sha+'\n')
(OUTPUT_DIR/'v6_input_manifest.json').write_text(json.dumps({'input_zip_name':INPUT_ZIP.name,'input_zip_sha256':hashlib.sha256(INPUT_ZIP.read_bytes()).hexdigest(),'baseline_sha256':baseline_sha,'source':'V5.1 generated artifacts','github_accessed':False},indent=2))
report = {'version':'V6','created_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'baseline_nodes':len(baseline['nodes']),'baseline_edges':len(baseline['edges']),'baseline_sha256':baseline_sha,'population_observations':len(observations),'unapproved_conceptual_links':len(concept_links),'automated_tests':tests,'tests_passed':sum(tests.values()),'tests_total':len(tests),'status':'PROTOTYPE_TESTS_PASS_SCIENTIFIC_REVIEW_PENDING','remaining_blockers':['Human clinical review of candidate mappings','Independent external epidemiological validation','Serum creatinine and laboratory cross-cycle comparability','Not a prescribing or market-sizing tool'],'no_github_access':True,'llm_used':False,'query_engine':'deterministic safety-gated evidence retrieval'}
(OUTPUT_DIR/'v6_validation_report.json').write_text(json.dumps(report,indent=2))
(OUTPUT_DIR/'README.txt').write_text('V6 research prototype. Deterministic guarded routing and retrieval; not an autonomous LLM agent, medical advice, or a treatment-eligibility estimator. Original clinical KG is read only.\n')
archive = pathlib.Path('Healthcare_KG_V6_Outputs.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUTPUT_DIR.iterdir()): z.write(p,p.name)
print(f"V6: {report['tests_passed']}/{report['tests_total']} tests passed; {len(observations)} population observations; {len(concept_links)} unapproved links. Output: {archive}")
for r in results: print(r['status'], '|', r['question'])


## Interactive question
Run this cell repeatedly with a new question. Answers are bounded by the supplied KG and population observations; the engine does not make external requests.

In [ ]:
question = 'How many US adults have CKD and diabetes?'
print(json.dumps(respond(question),indent=2))

## Output files
The execution cell writes `Healthcare_KG_V6_Outputs.zip` and a validation report. In Colab, download the ZIP with the optional cell below.

In [ ]:
# Optional Colab download:
# from google.colab import files
# files.download('Healthcare_KG_V6_Outputs.zip')